# Test local chunking b4 moving to Setonix

In [ ]:
from pathlib import Path
import warnings
import dask
from dask.distributed import Client
import xarray as xr

dpird_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_DPIRD_utc0_clean/DPIRD_final_stations_utc0.nc")
ecmwf_path= Path("C:/Users/John/OneDrive/Desktop/ICRAR/The_work/ICRAR-Weather-Forcasting/dataset_ecmwf_op_clean/2024/02/06.nc")

tmp_drop_path= Path("../.tmp/compressed")

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning
)

### DPIRD dataset

In [ ]:
dpird_ds=xr.open_dataset(dpird_path, engine="netcdf4")
chunked_dpird_ds= dpird_ds.chunk({'station':96,'time':52624})
chunked_dpird_ds

### ECMWF dataset

In [ ]:
ecmwf_ds=xr.open_dataset(ecmwf_path, engine="h5netcdf")
chunked_ecmwf_ds= ecmwf_ds.chunk({"time": 4})
chunked_ecmwf_ds

## Chunking & Compression
Compression step is in encoding

In [ ]:
client= Client(n_workers=2, threads_per_worker=2)
client

In [ ]:
import hdf5plugin
out_dir = tmp_drop_path
out_dir.mkdir(parents=True, exist_ok=True)

dpird_out = out_dir / "DPIRD_final_stations_utc0.chunked_zstd.nc"
ecmwf_out = out_dir / "2024_02_06.chunked_zstd.nc"

def build_var_encoding(ds, chunk_dict, complevel=9):
    enc = {}
    for v in ds.data_vars:
        var_chunks = tuple(chunk_dict.get(dim, ds[v].sizes[dim]) for dim in ds[v].dims)
        zstd_params = hdf5plugin.Zstd(clevel=complevel)

        enc[v] = {
            **zstd_params,
            "shuffle":True,
            "chunksizes": var_chunks,
        }
    return enc

dpird_chunks = {'station':96,'time':52624}
ecmwf_chunks = {"time": 4}
                
dpird_encoding = build_var_encoding(dpird_ds, dpird_chunks)
ecmwf_encoding = build_var_encoding(ecmwf_ds, ecmwf_chunks)

# Dask-lazy netCDF writes
dpird_write = dpird_ds.to_netcdf(
    path=dpird_out,
    engine="h5netcdf",
    encoding=dpird_encoding,
    compute=False,
)

ecmwf_write = ecmwf_ds.to_netcdf(
    path=ecmwf_out,
    engine="h5netcdf",
    encoding=ecmwf_encoding,
    compute=False,
)

# Execute both writes in parallel
dask.compute(dpird_write, ecmwf_write)

| Dataset | Original Size (uncompressed) | .to_netcdf() old chunks | .to_netcdf() new chunks
|---|---|---|---|
| DPIRD | 2.7GB | 563MB | 539MB |
| ECMWF | 4.0GB | 1.65GB | 1.60GB |

## A/B testing to before and after zstd to_netcdf()

In [ ]:
import h5py
og_dpird_path= dpird_path
og_ecmwf_path= ecmwf_path

new_dpird_path= Path(dpird_out)
new_ecmwf_path= Path(ecmwf_out)

def collect(path):
    rows = {}
    with h5py.File(path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows[name] = {
                    "shape": obj.shape,
                    "dtype": str(obj.dtype),
                    "chunks": obj.chunks,
                    "compression": obj.compression,
                    "compression_opts": obj.compression_opts,
                    "filters": dict(getattr(obj, "_filters", {}) or {}),
                    "shuffle": bool(obj.shuffle),
                    "storage": obj.id.get_storage_size(),
                }

        f.visititems(visit)
    return rows

def compare_storage(old_path, new_path, max_print=3):
    old = collect(old_path)
    new = collect(new_path)
    old_total = sum(v["storage"] for v in old.values())
    new_total = sum(v["storage"] for v in new.values())
    print(f"old dataset storage: {old_total:,}")
    print(f"new dataset storage: {new_total:,}")
    print(f"ratio: {new_total / old_total:.3f}x")

    flagged = []
    for name in sorted(set(old) & set(new)):
        o = old[name]
        n = new[name]
        if o["storage"] == 0:
            continue
        ratio = n["storage"] / o["storage"]
        if ratio > 1.05 or o["chunks"] != n["chunks"] or o["dtype"] != n["dtype"]:
            flagged.append((name, o, n, ratio))

    print(f"flagged datasets: {len(flagged)} (showing up to {max_print})")
    for i in range(min(max_print, len(flagged))):
        name, o, n, ratio = flagged[i]
        print(
            f"{name}: ratio={ratio:.3f}x "
            f"old_chunks={o['chunks']} new_chunks={n['chunks']} "
            f"old_dtype={o['dtype']} new_dtype={n['dtype']} "
            f"old_comp={o['compression']}/{o['compression_opts']} "
            f"new_comp={n['compression']}/{n['compression_opts']} "
            f"old_shuffle={o['shuffle']} new_shuffle={n['shuffle']} "
            f"new_filters={n['filters']}"
        )

compare_storage(og_dpird_path, new_dpird_path, max_print=3)
compare_storage(og_ecmwf_path, new_ecmwf_path, max_print=3)